# Session 01 — The Python Scientific Ecosystem & Tooling Setup

**Course:** Python for Optics  
**Duration:** ~2 hours (lecture + hands-on)  
**Prerequisites:** Pre-session setup guide completed (Python + VS Code installed)

---

## Learning Objectives

By the end of this session you will be able to:

1. Explain why Python dominates scientific computing and what its ecosystem looks like.
2. Navigate the Python interpreter, understand core data types, and use f-strings.
3. Manage environments with `conda` or `pip` + `venv`.
4. Work fluently in VS Code: run scripts, edit notebooks, and use essential shortcuts.
5. Write a short physics calculation both as a `.py` script and as a Jupyter notebook.

---

> 🔬 **Physics Insight:** Every code example in this course uses real physics — Gaussian beams, ultrashort pulses, structured light. No "hello world" detours.


## 1. Why Python for Physics?

Python has become the lingua franca of scientific computing. Here's why:

- **Rich ecosystem:** NumPy, SciPy, Matplotlib, and hundreds of domain-specific libraries.
- **Readability:** Code that reads like pseudocode (closer to plain english)— crucial when a PhD student inherits your simulation 3 years later.
- **Interoperability:** Bridges to C, Fortran, and GPU code when you need raw speed (Session 9).
- **Community:** From LIGO gravitational-wave analysis to quantum optics simulations, physicists publish and share Python code daily.

> 💡 **Pro Tip:** Python is *not* the fastest language. It's the fastest language *to get correct results in*. When a specific loop becomes a bottleneck, you optimize that loop — you don't rewrite everything in C. 

__Somebody said "Python is the second best language for everyting"__


### The Scientific Python Stack

| Library | Role | We'll cover in |
|---------|------|----------------|
| **NumPy** | N-dimensional arrays, linear algebra | Session 3 |
| **Matplotlib** | Publication-quality plotting | Session 4 |
| **SciPy** | Optimization, integration, FFT, signal processing | Session 5 |
| **Pandas** | Tabular / experimental data | Session 5 |
| **Jupyter** | Interactive notebooks | Today |
| **pytest** | Testing framework | Session 8 |

We'll also touch on Numba (JIT compilation), ctypes/f2py (C/Fortran bridges), and packaging tools later in the course.


## 2. Setup Recap

> If you followed the **Pre-Session Setup Guide**, you should already have:
> - Python 3.11+ (via Miniforge/Miniconda or python.org)
> - VS Code with extensions: **Python**, **Pylance**, **Jupyter**
> - A course environment: `conda activate physics-python` (or your virtualenv)

Let's verify everything works. Run the cell below — it should print with no errors:


In [1]:
import sys
print(f"Python version: {sys.version}")
print(f"Executable:     {sys.executable}")

# Quick check: can we import the scientific stack?
import importlib
for pkg in ["numpy", "matplotlib", "scipy"]:
    try:
        importlib.import_module(pkg)
        print(f"  ✓ {pkg} available")
    except ImportError:
        print(f"  ✗ {pkg} NOT FOUND — run: pip install {pkg}")


Python version: 3.11.15 (main, Mar 11 2026, 17:14:47) [Clang 20.1.8 ]
Executable:     /Applications/miniconda3/envs/physics-python/bin/python
  ✓ numpy available
  ✓ matplotlib available
  ✓ scipy available


> ⚠️ **Common Pitfall:** If the cell above shows a Python version you don't expect, your notebook kernel might be pointing to the wrong environment. In VS Code, click the kernel selector (top-right of the notebook) and choose the correct one.


## 3. The Python Interpreter — A Quick Tour

Python is a **dynamically typed**, **interpreted** language. You don't compile — you run.

There are three ways to execute Python code:

1. **Interactive REPL** — type `python` in a terminal. Great for quick tests.
2. **Script** — write a `.py` file, run it with `python my_script.py`.
3. **Notebook** — cells of code + text, executed interactively. What you're reading now.

In this course, we use **notebooks for exploration and learning**, and **.py files for reusable code** (functions, modules). You'll move between the two constantly.


## 4. Core Data Types

Python has a small set of built-in types that you'll use everywhere. Let's meet them with physics examples.

### 4.1 Numbers: `int` and `float`


In [2]:
# Integers — exact, arbitrary precision
num_pulses = 1_000_000          # underscores for readability
print(f"Number of pulses: {num_pulses}")
print(f"Type: {type(num_pulses)}")

# Floats — IEEE 754 double precision (~15 significant digits)
wavelength_nm = 800.0            # Ti:Sapphire central wavelength [nm]
wavelength_m = wavelength_nm * 1e-9
print(f"Wavelength: {wavelength_m:.2e} m")
print(f"Type: {type(wavelength_m)}")


Number of pulses: 1000000
Type: <class 'int'>
Wavelength: 8.00e-07 m
Type: <class 'float'>


> 🔬 **Physics Insight:** A `float` gives you ~15 decimal digits of precision. That's fine for most optics calculations, but be cautious when subtracting nearly equal numbers (catastrophic cancellation). We'll revisit this in Session 8 (Testing & Defensive Programming).


In [3]:
# Arithmetic operators
c = 3e8                # speed of light [m/s]
λ = 800e-9             # wavelength [m] — yes, Python 3 supports Unicode variable names!
ν = c / λ              # frequency [Hz]

print(f"Frequency: {ν:.4e} Hz")
print(f"Photon energy: {6.626e-34 * ν:.4e} J")
print(f"             = {6.626e-34 * ν / 1.602e-19:.2f} eV")


Frequency: 3.7500e+14 Hz
Photon energy: 2.4847e-19 J
             = 1.55 eV


> 💡 **Pro Tip:** Python allows Unicode identifiers like `λ`, `ν`, `θ`. Use them! They make physics code much more readable than `lam`, `nu`, `theta`. In VS Code, you can type these via OS character input or copy them opening `S01_symbol_shortcut_reference.html` in your browser. Also you can copy/paste them by double clicking this cell

Optics & EM: λ ν ω θ φ ψ ε μ χ η

Math & General Physics: α β γ δ Δ σ τ π ρ κ Σ Ω

### 4.2 Strings


In [5]:
laser_name = "Ti:Sapphire"
technique = 'CPA'          # single or double quotes — no difference

# String concatenation
full_name = laser_name + " " + technique + " system"
print(full_name)

# Length
print(f"Characters in name: {len(laser_name)}")


Ti:Sapphire CPA system
Characters in name: 11


### 4.3 f-strings (Formatted String Literals)

f-strings are Python's most readable way to embed values in text. Master them — you'll use them *constantly*.


In [8]:
λ_nm = 1030          # Yb:YAG wavelength [nm]
rep_rate_MHz = 80     # repetition rate [MHz]
pulse_energy_nJ = 5   # pulse energy [nJ]
avg_power_mW = rep_rate_MHz * pulse_energy_nJ  # P = f × E

print(f"Laser: λ = {λ_nm} nm, f_rep = {rep_rate_MHz} MHz")
print(f"Pulse energy: {pulse_energy_nJ} nJ")
print(f"Average power: {avg_power_mW} mW = {avg_power_mW / 1000:.1f} W")

# Format specifiers
print(f"--- Format specifiers ---")
print(f"Scientific: {avg_power_mW:.3e}")      # 4.000e+02
print(f"Fixed:      {avg_power_mW:.1f}")        # 400.0
print(f"Percentage: {0.85:.1%}")                # 85.0%
print(f"Padded:     {42:>10d}")                 # right-aligned


Laser: λ = 1030 nm, f_rep = 80 MHz
Pulse energy: 5 nJ
Average power: 400 mW = 0.4 W
--- Format specifiers ---
Scientific: 4.000e+02
Fixed:      400.0
Percentage: 85.0%
Padded:             42


⚡ **Try It:** Calculate the peak power of a laser pulse.

Given: pulse energy $E = 5\,\text{nJ}$, pulse duration $\tau_{FWHM} = 100\,\text{fs}$ (FWHM). 

> Assuming Gaussian shape and peak power $P_0$, $P(t)=P_0​exp(−t^2/2\sigma^2​)$, we compute $\tau_{FWHM}$ as $P(\tau_{FWHM}/2)=P_0/2$, therefore $\sigma = \frac{\tau_\text{FWHM}}{2\sqrt{2\ln 2}}$. 
>
> We compute the pulse energy as $E = \int_{-\infty}^{\infty} P(t)\, dt = P_0 \sqrt{2\pi} \sigma=P_0 \sqrt{2\pi} \sigma=P_0 \cdot \tau_\text{FWHM} \cdot \sqrt{\frac{\pi}{4\ln 2}}$. Hence $P_0 = \sqrt{\frac{4\ln 2}{\pi}}\cdot\frac{E}{\tau_\text{FWHM}} 
\approx \frac{0.94\, E}{\tau_\text{FWHM}}$

Print the result with appropriate units using an f-string.

<details><summary>Click to reveal solution</summary>

```python
E_nJ = 5
tau_fs = 100
E_J = E_nJ * 1e-9
tau_s = tau_fs * 1e-15

P_peak = 0.94 * E_J / tau_s

print(f"Peak power: {P_peak:.2e} W = {P_peak / 1e3:.1f} kW")
```

</details>


In [27]:
# Your code here:

E=5e-9
tau=100e-15
P0=0.94*E/tau
print(f"The peak power is {P0/1e3=} kW")


The peak power is P0/1e3=47.0 kW


### 4.4 Booleans

In [30]:
is_ultrafast = True
is_cw = False

# Comparison operators return booleans
pulse_duration_fs = 30
print(f"Sub-100 fs? {pulse_duration_fs < 100}")    # True
print(f"Few-cycle?  {pulse_duration_fs < 10}")      # False

# Boolean logic
high_energy = True
short_pulse = True
print(f"CPA candidate? {high_energy and short_pulse}")


Sub-100 fs? True
Few-cycle?  False
CPA candidate? True


### 4.5 `None` — The Absence of a Value


In [31]:
measurement = None  # No data yet

if measurement is None:
    print("No measurement recorded — detector offline?")
else:
    print(f"Measured value: {measurement}")


No measurement recorded — detector offline?


> ⚠️ **Common Pitfall:** Always use `is None` and `is not None`, never `== None`. The `is` operator checks identity, which is the correct test for `None`.


## 5. Collections — Lists, Tuples, Dicts

### 5.1 Lists — Ordered, Mutable


In [32]:
# A list of wavelengths in our lab [nm]
wavelengths = [532, 632.8, 800, 1030, 1550]
print(f"Available wavelengths: {wavelengths}")
print(f"Number of sources: {len(wavelengths)}")

# Indexing (0-based!)
print(f"First: {wavelengths[0]} nm")
print(f"Last:  {wavelengths[-1]} nm")

# Slicing
print(f"NIR sources: {wavelengths[2:]}")  # from index 2 onwards

# Lists are mutable, ie. we can change elements
wavelengths.append(405)        # add blue diode
wavelengths.sort()
print(f"Updated: {wavelengths}")


Available wavelengths: [532, 632.8, 800, 1030, 1550]
Number of sources: 5
First: 532 nm
Last:  1550 nm
NIR sources: [800, 1030, 1550]
Updated: [405, 532, 632.8, 800, 1030, 1550]


### 5.2 Tuples — Ordered, Immutable

Use tuples for fixed collections — things that shouldn't change after creation.


In [ ]:
# Laguerre-Gauss beam profiles

# A beam's transverse mode (l, p) — these are fixed once identified
mode = (1, 0)       # LG mode with l=1, p=0 (optical vortex)
l, p = mode          # tuple unpacking
print(f"LG mode: l={l}, p={p}")

# Tuples are immutable — this would raise TypeError:
# mode[0] = 2


LG mode: l=1, p=0


### 5.3 Dictionaries — Key-Value Pairs

Dictionaries are Python's most versatile data structure. Use them to label data.


In [43]:
# Laser parameters as a dictionary
laser = {
    "name": "Coherent Astrella",
    "wavelength_nm": 800,
    "rep_rate_kHz": 1,
    "pulse_duration_fs": 35,
    "pulse_energy_mJ": 7,
    "polarization": "linear horizontal",
}

print(f"Laser: {laser['name']}")
print(f"Energy per pulse: {laser['pulse_energy_mJ']} mJ")

# Add a derived quantity
laser["peak_power_GW"] = (
    0.94 * laser["pulse_energy_mJ"] * 1e-3 / (laser["pulse_duration_fs"] * 1e-15) / 1e9
)
print(f"Peak power: {laser['peak_power_GW']:.1f} GW")

# Iterate over key-value pairs
print("--- Full spec sheet ---")
for key, value in laser.items():
    print(f"  {key:>15s}: {value}")  # >15s: print string with length 15 an left justified 


Laser: Coherent Astrella
Energy per pulse: 7 mJ
Peak power: 188.0 GW
--- Full spec sheet ---
             name: Coherent Astrella
    wavelength_nm: 800
     rep_rate_kHz: 1
  pulse_duration_fs: 35
  pulse_energy_mJ: 7
     polarization: linear horizontal
    peak_power_GW: 188.0


⚡ **Try It:** Create a dictionary for a second laser in your lab (e.g., a Nd:YAG or a fiber laser). Include at least 4 parameters. Calculate and add a derived quantity (e.g., average power from rep rate × pulse energy). Print a formatted spec sheet.

<details><summary>Click to reveal solution</summary>

```python
fiber_laser = {
    "name": "IPG YLR-100",
    "type": "CW fiber",
    "wavelength_nm": 1070,
    "power_W": 100,
    "beam_quality_M2": 1.05,
}

# Derived: diffraction-limited beam waist at focus (f=100mm lens)
import math
f_mm = 100
λ_m = fiber_laser["wavelength_nm"] * 1e-9
M2 = fiber_laser["beam_quality_M2"]
D_mm = 5  # input beam diameter

w0_um = (4 * M2 * λ_m * (f_mm * 1e-3)) / (math.pi * D_mm * 1e-3) * 1e6
fiber_laser["focus_waist_um"] = round(w0_um, 1)

for k, v in fiber_laser.items():
    print(f"  {k:>25s}: {v}")
```

</details>


In [ ]:
# Your code here:

## 6. Variables, Assignment & Dynamic Typing

Python variables are **names** that point to objects. They don't have a fixed type — the *object* has a type.


In [44]:
x = 42          # x points to an int
print(f"x = {x}, type = {type(x)}")

x = 3.14        # now x points to a float — perfectly legal
print(f"x = {x}, type = {type(x)}")

x = "photon"    # now a string
print(f"x = {x}, type = {type(x)}")


x = 42, type = <class 'int'>
x = 3.14, type = <class 'float'>
x = photon, type = <class 'str'>


> ⚠️ **Common Pitfall:** Just because you *can* reassign a variable to a different type doesn't mean you *should*. In scientific code, if `wavelength` starts as a float, it should stay a float. We'll enforce this with type hints in Session 8.


In [45]:
# Multiple assignment
a, b, c = 1, 2, 3
print(f"a={a}, b={b}, c={c}")

# Swap — Pythonic way
a, b = b, a
print(f"After swap: a={a}, b={b}")


a=1, b=2, c=3
After swap: a=2, b=1


## 7. Operators

### 7.1 Arithmetic


In [46]:
# Standard arithmetic
print(f"5 + 3   = {5 + 3}")
print(f"5 - 3   = {5 - 3}")
print(f"5 * 3   = {5 * 3}")
print(f"5 / 3   = {5 / 3:.4f}")     # true division (always float)
print(f"5 // 3  = {5 // 3}")         # floor division
print(f"5 % 3   = {5 % 3}")          # modulo
print(f"5 ** 3  = {5 ** 3}")         # power

# Useful for physics: integer division for binning detector pixels
pixel = 1247
bin_size = 4
bin_index = pixel // bin_size
print(f"Pixel {pixel} → bin {bin_index}")


5 + 3   = 8
5 - 3   = 2
5 * 3   = 15
5 / 3   = 1.6667
5 // 3  = 1
5 % 3   = 2
5 ** 3  = 125
Pixel 1247 → bin 311


### 7.2 Comparison & Membership


In [47]:
# Comparison
print(f"800 > 400:  {800 > 400}")    # True — NIR vs visible
print(f"800 == 800: {800 == 800}")   # True

# Membership
nir_sources = [800, 1030, 1064, 1550]
print(f"1030 in NIR? {1030 in nir_sources}")   # True
print(f"532 in NIR?  {532 in nir_sources}")     # False


800 > 400:  True
800 == 800: True
1030 in NIR? True
532 in NIR?  False


## 8. Basic Input/Output

### `print()` — You've already been using it

### `input()` — Read from the user (rarely used in scientific code, but good to know)


In [49]:
# In a notebook, input() pops up a text box in the upper side of your window
# Uncomment and run if you want to try:

# name = input("Enter your laser's name: ")
# print(f"You entered: {name}")

# In practice, we parameterize with variables or config files, not interactive input.


### The `math` module — Basic math without NumPy


In [52]:
import math

# Gaussian beam divergence: θ = λ / (π w₀)
w0_mm = 1.0                              # beam waist [mm]
λ_um = 0.8                               # wavelength [μm]

w0_m = w0_mm * 1e-3
λ_m = λ_um * 1e-6

theta_rad = λ_m / (math.pi * w0_m)
theta_mrad = theta_rad * 1e3

print(f"Beam waist:    w₀ = {w0_mm} mm")
print(f"Wavelength:    λ  = {λ_um} μm")
print(f"Divergence:    θ  = {theta_mrad:.3f} mrad")
print(f"               θ  = {math.degrees(theta_rad):.4f}°")

# Rayleigh range
z_R = math.pi * w0_m**2 / λ_m
print(f"Rayleigh range: z_R = {z_R:.3f} m = {z_R*100:.1f} cm")


Beam waist:    w₀ = 1.0 mm
Wavelength:    λ  = 0.8 μm
Divergence:    θ  = 0.255 mrad
               θ  = 0.0146°
Rayleigh range: z_R = 3.927 m = 392.7 cm


> 🔬 **Physics Insight:** The `math` module works on single numbers. For arrays (which is how you'll do most physics), you'll use `numpy` (Session 3). The API is almost identical: `math.sin(x)` → `numpy.sin(array)`.


## 9. VS Code — Your Daily Driver

VS Code is a free, extensible editor that handles everything from quick scripts to large codebases. Here's what matters on day one.

### 9.1 Extensions We Installed

| Extension | What it does |
|-----------|-------------|
| **Python** (ms-python) | Language support, linting, run/debug |
| **Pylance** | Fast autocomplete, type checking |
| **Jupyter** | Notebook editing inside VS Code |

> More extensions will be added in later sessions (Error Lens in S02, GitLens in S07).

### 9.2 Essential Shortcuts

| Action | Windows/Linux | macOS |
|--------|--------------|-------|
| Command Palette | `Ctrl+Shift+P` | `⌘+Shift+P` |
| Open terminal | `` Ctrl+` `` | `` Ctrl+` `` |
| Run selection/cell | `Shift+Enter` | `Shift+Enter` |
| Go to definition | `F12` | `F12` |
| Quick fix | `Ctrl+.` | `⌘+.` |
| Toggle sidebar | `Ctrl+B` | `⌘+B` |
| Find in files | `Ctrl+Shift+F` | `⌘+Shift+F` |
| Multi-cursor | `Alt+Click` | `⌥+Click` |

> 💡 **Pro Tip:** Invest 5 minutes learning these shortcuts now. Over a semester, they save *hours*.

### 9.3 Jupyter in VS Code vs. Browser

| Feature | VS Code | Browser (JupyterLab) |
|---------|---------|---------------------|
| Autocomplete | Excellent (Pylance) | Basic |
| Debugging cells | ✓ (built-in) | Limited |
| Git integration | ✓ | ✗ |
| Variable explorer | ✓ | ✓ |
| Terminal access | ✓ (same window) | Separate tab |
| Remote (SSH) | ✓ (one click) | Manual tunnel |

**Our recommendation:** Use VS Code for everything. Fall back to JupyterLab only if you need its drag-and-drop cell rearrangement or specific JupyterLab extensions.


## 10. Managing Python Environments

### Why environments?

Your beam propagation code needs NumPy 1.26. Your collaborator's code needs NumPy 1.24. Without environments, you're stuck.

An **environment** is an isolated Python installation with its own packages. Each project gets its own environment.

### 10.1 Option A: Conda (Recommended)

```bash
# Create an environment
conda create -n physics-python python=3.11 numpy scipy matplotlib jupyter

# Activate it
conda activate physics-python

# Install more packages
conda install pandas h5py

# List installed packages
conda list

# Deactivate
conda deactivate
```

### 10.2 Option B: pip + venv

```bash
# Create a virtual environment
python -m venv .venv

# Activate it
source .venv/bin/activate        # Linux/macOS
.venv\Scripts\activate           # Windows

# Install packages
pip install numpy scipy matplotlib jupyter

# Freeze requirements
pip freeze > requirements.txt
```

> 💡 **Pro Tip:** Whichever you choose, **always** work inside an environment. Never install packages into your system Python. If your VS Code terminal shows `(base)` or `(physics-python)` in the prompt, you're good.


⚡ **Try It:** Open a terminal in VS Code (`` Ctrl+` ``), verify your active environment, and check which packages are installed. Can you find NumPy's version?

<details><summary>Click to reveal solution</summary>

```bash
# Conda users:
conda list numpy

# pip users:
pip show numpy

# Or from Python:
python -c "import numpy; print(numpy.__version__)"
```

</details>


## 11. Notebook vs. Script — When to Use Which

| Use a **notebook** when... | Use a **script** (`.py`) when... |
|---|---|
| Exploring data interactively | Building reusable functions/modules |
| Prototyping a calculation | Running automated pipelines |
| Creating a report or tutorial | Code will be imported elsewhere |
| Visualizing results inline | Performance matters |

In practice, the workflow is:

1. **Explore** in a notebook — try things, plot, iterate.
2. **Extract** working code into a `.py` module.
3. **Import** the module back into the notebook (or a script).

We'll practice this "notebook → module" workflow throughout the course.


### Let's try both

Below is a complete physics calculation done in this notebook. After the notebook, you'll find the same calculation as a `.py` script in the exercises.


In [53]:
# === Gaussian pulse: time-bandwidth product check ===
# A transform-limited Gaussian pulse satisfies: Δν·Δt = 0.44

import math

# Given parameters
tau_fs = 50        # FWHM pulse duration [fs]
dlambda_nm = 20    # FWHM spectral bandwidth [nm]
lambda0_nm = 800   # central wavelength [nm]

# Convert spectral bandwidth to frequency bandwidth
c = 3e8
lambda0_m = lambda0_nm * 1e-9
dlambda_m = dlambda_nm * 1e-9

# Δν = c·Δλ/λ² (from ν = c/λ, differentiate)
dnu = c * dlambda_m / lambda0_m**2   # [Hz]
tau_s = tau_fs * 1e-15                # [s]

# Time-bandwidth product
TBP = dnu * tau_s

print("=" * 50)
print("  Gaussian Pulse — Time-Bandwidth Product")
print("=" * 50)
print(f"  Central wavelength:  {lambda0_nm} nm")
print(f"  Spectral bandwidth:  {dlambda_nm} nm = {dnu:.3e} Hz")
print(f"  Pulse duration:      {tau_fs} fs")
print(f"  TBP = Δν·Δt:        {TBP:.3f}")
print(f"  Transform limit:     0.441 (Gaussian)")
print(f"  Chirp factor:        {TBP / 0.441:.2f}×")
if TBP / 0.441 > 1.1:
    print("  → Pulse is CHIRPED")
else:
    print("  → Pulse is near transform-limited")
print("=" * 50)


  Gaussian Pulse — Time-Bandwidth Product
  Central wavelength:  800 nm
  Spectral bandwidth:  20 nm = 9.375e+12 Hz
  Pulse duration:      50 fs
  TBP = Δν·Δt:        0.469
  Transform limit:     0.441 (Gaussian)
  Chirp factor:        1.06×
  → Pulse is near transform-limited


## 12. Type Conversions

Sometimes you need to explicitly convert between types.


In [ ]:
# int ↔ float
n_photons_float = 1e6
n_photons_int = int(n_photons_float)
print(f"Float: {n_photons_float}, Int: {n_photons_int}")

# str → number (e.g., reading from a file)
reading = "3.1415"
value = float(reading)
print(f"String '{reading}' → float {value}")

# number → str (for display/filenames)
run_number = 42
filename = "data_run_" + str(run_number).zfill(4) + ".csv"
print(f"Filename: {filename}")

# Better: use f-string directly
filename = f"data_run_{run_number:04d}.csv"
print(f"Filename: {filename}")


> ⚠️ **Common Pitfall:** `int()` truncates, it does NOT round:
> ```python
> int(2.9)   # → 2, not 3
> round(2.9)  # → 3
> ```


## 13. Getting Help

Three ways to get help without leaving your editor:


In [ ]:
# 1. help() — detailed documentation
# help(math.sqrt)  # Uncomment to see full docs

# 2. ? in Jupyter (same as help but formatted)
# math.sqrt?       # Uncomment in a notebook cell

# 3. type() — what IS this thing?
x = 3.14
print(type(x))            # <class 'float'>
print(type([1, 2, 3]))    # <class 'list'>
print(type({"a": 1}))     # <class 'dict'>


> 💡 **Pro Tip:** In VS Code, hover over any function to see its signature and docstring. Press `F12` to jump to its source code. This is faster than Googling.


## 14. Putting It All Together — Laser Lab Inventory

Let's combine everything from this session into a mini-exercise right here.


In [55]:
# Laser lab inventory — combining types, collections, f-strings, math

import math

lab_lasers = [
    {
        "name": "Ti:Sapphire (Astrella)",
        "wavelength_nm": 800,
        "rep_rate_kHz": 1,
        "pulse_energy_mJ": 7.0,
        "pulse_duration_fs": 35,
    },
    {
        "name": "Yb:fiber oscillator",
        "wavelength_nm": 1030,
        "rep_rate_MHz": 80,
        "pulse_energy_nJ": 5.0,
        "pulse_duration_fs": 100,
    },
    {
        "name": "HeNe alignment laser",
        "wavelength_nm": 632.8,
        "cw_power_mW": 5.0,
    },
]

print("=" * 65)
print(f"  {'LASER LAB INVENTORY':^61}")
print("=" * 65)

for laser in lab_lasers:
    print(f"{laser['name']}")
    print(f"     λ = {laser['wavelength_nm']} nm")
    
    # Check if it's pulsed or CW
    if "pulse_duration_fs" in laser:
        τ = laser["pulse_duration_fs"]
        print(f"     Pulse duration: {τ} fs")
        
        # Calculate peak power depending on available units
        if "pulse_energy_mJ" in laser:
            E_J = laser["pulse_energy_mJ"] * 1e-3
            f_Hz = laser["rep_rate_kHz"] * 1e3
            P_avg = E_J * f_Hz
            print(f"     Rep rate: {laser['rep_rate_kHz']} kHz")
            print(f"     Pulse energy: {laser['pulse_energy_mJ']} mJ")
            print(f"     Average power: {P_avg:.1f} W")
        elif "pulse_energy_nJ" in laser:
            E_J = laser["pulse_energy_nJ"] * 1e-9
            f_Hz = laser["rep_rate_MHz"] * 1e6
            P_avg = E_J * f_Hz
            print(f"     Rep rate: {laser['rep_rate_MHz']} MHz")
            print(f"     Pulse energy: {laser['pulse_energy_nJ']} nJ")
            print(f"     Average power: {P_avg*1e3:.0f} mW")
        
        P_peak = 0.94 * E_J / (τ * 1e-15)
        print(f"     Peak power: {P_peak:.2e} W")
    else:
        print(f"     CW power: {laser['cw_power_mW']} mW")

print("" + "=" * 65)


                       LASER LAB INVENTORY                     
Ti:Sapphire (Astrella)
     λ = 800 nm
     Pulse duration: 35 fs
     Rep rate: 1 kHz
     Pulse energy: 7.0 mJ
     Average power: 7.0 W
     Peak power: 1.88e+11 W
Yb:fiber oscillator
     λ = 1030 nm
     Pulse duration: 100 fs
     Rep rate: 80 MHz
     Pulse energy: 5.0 nJ
     Average power: 400 mW
     Peak power: 4.70e+04 W
HeNe alignment laser
     λ = 632.8 nm
     CW power: 5.0 mW


## Summary

| Concept | Key takeaway |
|---------|-------------|
| **Python ecosystem** | NumPy + SciPy + Matplotlib = your daily toolkit |
| **Data types** | `int`, `float`, `str`, `bool`, `None` |
| **Collections** | `list` (mutable), `tuple` (immutable), `dict` (key-value) |
| **f-strings** | `f"λ = {wavelength:.2f} nm"` — your best friend for output |
| **Environments** | Always use conda or venv. Never pollute system Python |
| **VS Code** | Command Palette (`Ctrl+Shift+P`) is the gateway to everything |
| **Notebook vs. script** | Explore in notebooks, build in `.py` files |

### What's Next

**Session 2:** Control flow (if/else, loops, comprehensions), functions, and the VS Code debugger. We'll start writing reusable physics code.

---

> 📝 **Now open `S01_exercises.ipynb`** for the hands-on practice problems.
